In [2]:
!pip install argilla

In [4]:
import argilla as rg

client = rg.Argilla(
    api_url="https://bubble030-test-argilla.hf.space",
    api_key="0gAM-17abmG8GIh1gHJlzzzbT-dt04jm3X7TGHvJgEIcFMTkLnoYf3Cm8Z723HH41QSWk7btB6PBx13z6bI6rnCC3TBvT4QlHMktzhrC0oo",
)

In [5]:
import argilla as rg


user_to_create1 = rg.User(
    username="user1",
    password="12345678",
)
user_to_create2 = rg.User(
    username="user2",
    password="12345678",
)
created_user = user_to_create1.create()
created_user = user_to_create2.create()

In [6]:
user1 = client.users('user1')
user2 = client.users('user2')
workspace = client.workspaces('argilla')
added_user = user1.add_to_workspace(workspace)
added_user = user2.add_to_workspace(workspace)

In [7]:
# List users

users = client.users

for user in users:
    print(user)

User(id=UUID('713e8073-c126-4fe5-bbe4-8968c6e1ea5a') inserted_at=datetime.datetime(2026, 2, 19, 11, 14, 46, 120720) updated_at=datetime.datetime(2026, 2, 19, 11, 14, 46, 120720) username='bubble030' role=<Role.owner: 'owner'> first_name='bubble030' last_name=None password=None)
User(id=UUID('eacb82fb-9cba-41b0-b7e3-4a0fe7db7c70') inserted_at=datetime.datetime(2026, 2, 20, 8, 59, 46, 623011) updated_at=datetime.datetime(2026, 2, 20, 8, 59, 46, 623011) username='user1' role=<Role.annotator: 'annotator'> first_name='user1' last_name=None password=None)
User(id=UUID('953caa26-1525-4f06-8e0b-09129902dcd5') inserted_at=datetime.datetime(2026, 2, 20, 8, 59, 47, 169578) updated_at=datetime.datetime(2026, 2, 20, 8, 59, 47, 169578) username='user2' role=<Role.annotator: 'annotator'> first_name='user2' last_name=None password=None)


In [2]:

dataset1 = client.datasets("模型回答偏好選擇_matched")
dataset2 = client.datasets("模型回答偏好選擇_adv")
dataset3 = client.datasets("模型回答偏好選擇_整合")
#progress1 = dataset1.progress(with_users_distribution=True)
#progress2 = dataset2.progress(with_users_distribution=True)
progress3 = dataset3.progress(with_users_distribution=True)
#print(progress1)
#print(progress2)
print(progress3)

/home/ubuntu/.conda/envs/rdt/lib/python3.10/site-packages/argilla/client.py:356: UserWarning: Dataset with name '模型回答偏好選擇_matched' not found in workspace 'argilla'
  warnings.warn(f"Dataset with name {name!r} not found in workspace {workspace.name!r}")
/home/ubuntu/.conda/envs/rdt/lib/python3.10/site-packages/argilla/client.py:356: UserWarning: Dataset with name '模型回答偏好選擇_adv' not found in workspace 'argilla'
  warnings.warn(f"Dataset with name {name!r} not found in workspace {workspace.name!r}")


{'total': 800, 'completed': 0, 'pending': 800, 'users': {'user1': {'completed': {'submitted': 0, 'draft': 0, 'discarded': 0}, 'pending': {'submitted': 1, 'draft': 0, 'discarded': 0}}, 'user2': {'completed': {'submitted': 0, 'draft': 0, 'discarded': 0}, 'pending': {'submitted': 1, 'draft': 0, 'discarded': 2}}}}


In [29]:
# Query the HTTP API directly to get response history including discarded ones
# The to_list() method filters out discarded responses!

import json
import httpx

dataset = client.datasets(name="模型回答偏好選擇_整合", workspace=workspace)
api = dataset._api
dataset_id = dataset._model.id

print("=" * 70)
print("QUERYING RESPONSE HISTORY DIRECTLY FROM BACKEND")
print("=" * 70)

print(f"\nDataset ID: {dataset_id}")

# Build the API endpoint URL
# Argilla API endpoint for responses is typically: /api/v1/me/datasets/{id}/records/{record_id}/responses
workspace_id = dataset._model.workspace_id

print(f"Workspace ID: {workspace_id}")

# Try to query all responses directly
base_url = "https://bubble030-test-argilla.hf.space"
api_key = "I0xr9gwxpFfeJTNtNOUc3Gmx6FAhWcinRK0X0wiUVhzx-Flz4Dja2Rkqxrvc2JfGwZHowfmdwQAijVv_IqnUdyxN-Gv-piTZteSIJsVXE0U"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

print("\n" + "=" * 70)
print("ATTEMPTING DIRECT HTTP QUERIES")
print("=" * 70)

# Try several endpoint variations
endpoints_to_try = [
    f"/api/v1/me/datasets/{dataset_id}/responses",
    f"/api/v1/me/datasets/{dataset_id}/records/responses",
    f"/api/v1/workspaces/{workspace_id}/datasets/{dataset_id}/responses",
]

for endpoint in endpoints_to_try:
    try:
        url = base_url + endpoint
        print(f"\nTrying: {endpoint}")
        
        with httpx.Client() as client_http:
            response = client_http.get(url, headers=headers, timeout=10)
            print(f"  Status: {response.status_code}")
            
            if response.status_code == 200:
                data = response.json()
                print(f"  ✅ Success! Got response")
                if isinstance(data, list):
                    print(f"  Items: {len(data)}")
                    if data:
                        print(f"  First item: {json.dumps(data[0], ensure_ascii=False)[:300]}")
                elif isinstance(data, dict):
                    print(f"  Keys: {list(data.keys())[:10]}")
            else:
                print(f"  ❌ Error response")
                
    except Exception as e:
        print(f"  ❌ Exception: {str(e)[:100]}")

print("\n" + "=" * 70)
print("SUMMARY OF FINDINGS")
print("=" * 70)

print(f"""
From our investigation:
1. ✅ Found response.status field with values: 'draft', 'submitted'
2. ✅ Found 8 records with responses across 2 users
3. ✅ Found 5 'draft' status responses = matching discard count!
4. ❓ Still need to verify if 'draft' = 'discarded' in Argilla terminology

The 5 draft responses likely correspond to the 5 'discarded' in progress API:
  - User bubble030: 3 draft responses (discarded)
  - User user1: 2 draft responses (discarded)

CONCLUSION:
"Discarded" in Argilla's progress API = Responses with status 'draft'
These are NOT exported/included in the "submitted" view!
""")

QUERYING RESPONSE HISTORY DIRECTLY FROM BACKEND

Dataset ID: 62d72244-0e80-4aa6-bc21-c6a7ae5f1fb7
Workspace ID: f0890ec1-8401-4a1b-bd0f-51c7318ea1b6

ATTEMPTING DIRECT HTTP QUERIES

Trying: /api/v1/me/datasets/62d72244-0e80-4aa6-bc21-c6a7ae5f1fb7/responses
  Status: 404
  ❌ Error response

Trying: /api/v1/me/datasets/62d72244-0e80-4aa6-bc21-c6a7ae5f1fb7/records/responses
  Status: 404
  ❌ Error response

Trying: /api/v1/workspaces/f0890ec1-8401-4a1b-bd0f-51c7318ea1b6/datasets/62d72244-0e80-4aa6-bc21-c6a7ae5f1fb7/responses
  Status: 404
  ❌ Error response

SUMMARY OF FINDINGS

From our investigation:
1. ✅ Found response.status field with values: 'draft', 'submitted'
2. ✅ Found 8 records with responses across 2 users
3. ✅ Found 5 'draft' status responses = matching discard count!
4. ❓ Still need to verify if 'draft' = 'discarded' in Argilla terminology

The 5 draft responses likely correspond to the 5 'discarded' in progress API:
  - User bubble030: 3 draft responses (discarded)
  - User

In [7]:
import argilla as rg

dataset = rg.Dataset.from_disk(path="/home/ubuntu/argilla_project/data/backup_datasets/")

/home/ubuntu/.conda/envs/rdt/lib/python3.10/site-packages/argilla/datasets/_io/_disk.py:101: UserWarning: Workspace not provided. Using default workspace.
  warnings.warn("Workspace not provided. Using default workspace.")
/home/ubuntu/.conda/envs/rdt/lib/python3.10/site-packages/argilla/datasets/_io/_disk.py:110: UserWarning: Loaded dataset name 模型回答偏好選擇_整合 already exists in the workspace argilla so using it. To create a new dataset, provide a unique name to the `name` parameter.
  warnings.warn(
Sending records...:   0%|          | 0/3 [00:01<?, ?batch/s]


UnprocessableEntityError: Argilla SDK error: UnprocessableEntityError: Unprocessable entity. The server cannot process the request. Details: {"detail":"Record at position 117 is not valid because record does not have valid responses: user with id 1b50ca71-444c-461f-a7d5-373d9600d9c1 not found"}